# ¿Ofensivo o defensivo? — Jardine / América

Posesión %, field tilt, PPDA, tiros/xG a favor y en contra, e índice de directividad (qué tan directo/elaborado juega el equipo).


In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import matplotlib.pyplot as plt
from framework_analitico import (
    normalizar_direccion_ataque,
    perfil_ofensivo_defensivo,
    indice_directividad,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

# Normalizar dirección de ataque ANTES del perfil (field tilt y PPDA usan x normalizada)
events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)


## 1. Perfil ofensivo/defensivo completo

Compara a América contra el conjunto de sus rivales, dentro del mismo dataset de 129 partidos (nunca contra la liga completa, para no mezclar competiciones o épocas distintas).


In [ ]:
perfil = perfil_ofensivo_defensivo(events_norm, equipo=EQUIPO)

print(f"Posesión: {perfil['posesion_pct']}%")
print(f"Field tilt (% de eventos propios en el tercio de ataque): {perfil['field_tilt_pct']}%")
print(f"Field tilt relativo (estándar de industria, equipo vs. rival en sus tercios de ataque): {perfil['field_tilt_relativo_pct']}%")
print(f"PPDA (pases rivales permitidos por acción defensiva propia, x>{perfil['umbral_ppda_x']}): {perfil['ppda']}")
print(f"Tiros a favor: {perfil['tiros_favor']}  |  Tiros en contra: {perfil['tiros_contra']}")
if "xg_favor" in perfil:
    print(f"xG a favor: {perfil['xg_favor']}  |  xG en contra: {perfil['xg_contra']}")
    print(f"Diferencia de xG: {perfil['xg_favor'] - perfil['xg_contra']:+.2f}")


**Nota de cautela sobre el PPDA:** aun ajustando la zona al 40% final del campo (x>72, el estándar de industria), el PPDA sale muy por debajo de los benchmarks publicados (8-15 típico; aquí ~2-2.5). La causa probable es que StatsBomb registra el evento `Duel` de forma mucho más granular que otros proveedores (ej. Opta, la fuente de la mayoría de los benchmarks públicos), inflando el conteo de "acciones defensivas". **Por eso este PPDA no se compara contra benchmarks externos** — solo es válido de forma relativa, entre partidos o entre equipos calculados con esta misma función y la misma fuente de datos. El dato más confiable y defendible del bloque es el **field tilt relativo (49.9%)**: casi exactamente equilibrado con los rivales.

## 2. Índice de directividad: ¿qué tan directo o elaborado juega el equipo?

A diferencia de `salida_corta_vs_larga` (que solo mira la salida desde el tercio propio), esto mide el % de pases largos sobre TODOS los pases del equipo, en cualquier zona del campo.


In [ ]:
directividad = indice_directividad(events_norm, equipo=EQUIPO)

print(f"Total de pases analizados: {directividad['total_pases']}")
print(f"% de pases largos (≥18m): {directividad['pct_pases_largos']}%")
print(f"Longitud promedio de pase: {directividad['longitud_promedio_m']}m")


## 3. Comparación visual: tiros y xG a favor vs. en contra


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].bar(["Tiros a favor", "Tiros en contra"], [perfil["tiros_favor"], perfil["tiros_contra"]],
           color=["#27ae60", "#c0392b"])
axs[0].set_title("Tiros")

if "xg_favor" in perfil:
    axs[1].bar(["xG a favor", "xG en contra"], [perfil["xg_favor"], perfil["xg_contra"]],
               color=["#27ae60", "#c0392b"])
    axs[1].set_title("xG")

fig.suptitle(f"{EQUIPO}: a favor vs. en contra (129 partidos)")
plt.tight_layout()
plt.savefig("../outputs/figures/favor_vs_contra.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Síntesis: ¿ofensivo o defensivo?

**Resultados:** posesión 50.8%, field tilt relativo 49.9% (prácticamente equilibrado), xG a favor 206.21 vs. xG en contra 120.67 (**+85.54 de diferencia**, el dato más contundente del bloque). PPDA reportado con nota de cautela (ver arriba), no comparable a benchmarks externos.

**Lectura:** América no es un equipo que domine el terreno de forma aplastante (posesión y field tilt relativo rondan el 50%), pero sí convierte ese terreno compartido en una ventaja de calidad de ocasiones muy superior a la de sus rivales. Es un perfil de equipo **eficiente más que dominante territorialmente** — coherente con lo visto en fase ofensiva (construcción mixta, poco contragolpe pero muy eficiente cuando aparece) y fase defensiva (presión puntual en zona media, no presión alta sistemática). Con esto queda cubierta la pregunta 4 del Plan Maestro.
